In [3]:
import pandas as pd
import numpy as np
df_j = pd.read_parquet('../data/jaccard_overall.parquet')
df_p = pd.read_parquet('../data/researchers_publications.parquet')
df_u = pd.read_parquet('../data/umap_positions_with_source_id.parquet')
"""
    Calcule la distance entre deux chercheurs via barycentres pondérés.
    - jaccard_fp: matrice distance journaux (ISSN/EISSN).
    - pubs_fp: publications des chercheurs.
    - umap_fp: mapping source-id → ISSN/EISSN.
    """

    
    # 2. Charger publications et mapping
    
    # mapping source-id → ISSN/EISSN
mapping = df_u.set_index('source-id')['id'].astype(str).to_dict()
def get_weights(author):
        row = df_p[df_p['author_name'] == author]
        if row.empty:
            raise KeyError(f"Auteur '{author}' non trouvé")
        journals = row.iloc[0]['journals']
        issn_counts = []
        for j in journals:
            sid = str(j['journal_id'])
            issn = mapping.get(sid)
            if issn and issn in df_j.index:
                issn_counts.append((issn, j['count']))
        if not issn_counts:
            raise ValueError(f"Aucun journal valide pour '{author}'")
        ids, counts = zip(*issn_counts)
        w = np.array(counts, dtype=float)
        w /= w.sum()
        return list(ids), w


def researcher_distance(author1, author2,
                        jaccard_fp='../data/jaccard_overall.parquet',
                        pubs_fp='../data/researchers_publications.parquet',
                        umap_fp='../data/umap_positions_with_source_id.parquet'):
    
   

    # 3. Récupérer IDs et poids
    ids1, w1 = get_weights(author1)
    ids2, w2 = get_weights(author2)

    # 4. Construire la liste unique d'ISSN et extraire sous-matrice
    combined = []
    for _id in ids1 + ids2:
        if _id not in combined:
            combined.append(_id)
    M = df_j.loc[combined, combined].astype(float).values
    D2 = M**2

    # 5. Indices relatifs pour each researcher
    a_idx = [combined.index(i) for i in ids1]
    b_idx = [combined.index(i) for i in ids2]

    # 6. Extraire sous-blocs et calculs
    D2_ab = D2[np.ix_(a_idx, b_idx)]
    D2_aa = D2[np.ix_(a_idx, a_idx)]
    D2_bb = D2[np.ix_(b_idx, b_idx)]
    term_ab = w1 @ D2_ab @ w2
    term_aa = w1 @ D2_aa @ w1
    term_bb = w2 @ D2_bb @ w2

    # 7. Distance finale
    d2 = term_ab - 0.5 * term_aa - 0.5 * term_bb
    return float(np.sqrt(max(d2, 0.0)))

# Exemple d'utilisation :
dist = researcher_distance("Giroire F.", "Nisse N.")
print(f"Distance : {dist:.3f}")
dist = researcher_distance("Giroire H.", "Nisse N.")
print(f"Distance : {dist:.3f}")
dist = researcher_distance("Giroire J.", "Nisse N.")
print(f"Distance : {dist:.3f}")
dist = researcher_distance("Giroire B.", "Nisse N.")
print(f"Distance : {dist:.3f}")

Distance : 0.185
Distance : 0.395
Distance : 0.474
Distance : 0.455


In [2]:
import pandas as pd

df = pd.read_parquet("../data/umap_positions_with_source_id.parquet")
mapping = df[["source-id", "id"]].rename(columns={"id": "issn"})
mapping.to_parquet("../data/journal_mapping.parquet", index=False)